In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import requests
import pymongo
import time

# 구군별 스타벅스 지점을 MongoDB에 저장하기

## 매장 정보를 얻어오기 위해서 시도코드와 시도코드에 따른 구군코드를 얻어오기 위해 MongoDB에 저장된 데이터를 얻어온다.

In [2]:
conn = pymongo.MongoClient('mongodb://localhost:27017') # MongoDB와 연결한다.
starbucks = conn['starbucks'] # MongoDB에 연결된 상태에서 starbucks 데이터베이스와 연결한다.
gugunAll = starbucks['gugunAll'] # starbucks 데이터베이스와 연결된 상태에서 gugunAll 컬렉션에 연결한다.
gugunList = gugunAll.find({}, {'_id': 0}) # starbucks> db.gugunAll.find({}, {'_id': 0})
print(gugunList)

# for gugun in gugunList:
    # print(gugun)

In [3]:
sidoSet = set()
for gugun in gugunList:
    sidoSet.add(gugun['sido_nm'])
print(sidoSet)

{'경기', '제주', '대전', '울산', '부산', '전남광주', '서울', '충북', '세종', '인천', '경북대구통합특별시', '충남', '강원', '경남', '전북'}


## 시도 이름과 구군 이름을 입력받아 매장 정보를 얻어와서 MongoDB에 저장할 도큐먼트를 만들어 저장한다.

시도 이름을 보여주고 매장 정보를 검색할 시도 이름을 입력받는다.

In [4]:
print('시도 이름')
for sido in sidoSet:
    print(f'[{sido}]', end=' ')
print()
sido_nm = input('매장 정보를 검색하려는 시도 이름을 입력하세요: ')

시도 이름
[경기] [제주] [대전] [울산] [부산] [전남광주] [서울] [충북] [세종] [인천] [경북대구통합특별시] [충남] [강원] [경남] [전북] 
매장 정보를 검색하려는 시도 이름을 입력하세요: 경기


입력받은 시도 이름에 따른 구군 이름을 보여주고 검색할 구군 정보를 얻어온다.

In [5]:
gugunList = gugunAll.find({'sido_nm': sido_nm}, {'_id': 0}) # starbucks> db.gugunAll.find({'sido_nm': '서울'}, {'_id': 0})
print('구군 이름')
for gugun in gugunList:
    # print(gugun)
    print(f'[{gugun["gugun_nm"]}]', end=' ')
print()
gugun_nm = input('매장 정보를 검색하려는 구군 이름을 입력하세요: ')

구군 이름
[가평군] [고양시 덕양구] [고양시 일산동구] [고양시 일산서구] [과천시] [광명시] [광주시] [구리시] [군포시] [김포시] [남양주시] [동두천시] [부천시] [성남시 분당구] [성남시 수정구] [성남시 중원구] [수원시 권선구] [수원시 영통구] [수원시 장안구] [수원시 팔달구] [시흥시] [안산시 단원구] [안산시 상록구] [안성시] [안양시 동안구] [안양시 만안구] [양주시] [양평군] [여주시] [연천군] [오산시] [용인시 기흥구] [용인시 수지구] [용인시 처인구] [의왕시] [의정부시] [이천시] [파주시] [평택시] [포천시] [하남시] [화성시] 
매장 정보를 검색하려는 구군 이름을 입력하세요: 성남


구군 코드를 사용하면 아래의 코드처럼 간단하게 검색할 수 있지만 우리가 입력받은 정보는 `시도 이름`과 `구군 이름`밖에 없기 때문에 사용할 수 없다.
`gugunAll.find({'gugun_cd': gugun['gugun_cd']}, {'_id': 0}).next() # starbucks> db.gugunAll.find({'gugun_cd': '0101'}, {'_id': 0})`

사용하는 조건이 `eq`와 `and` 조건일 경우 아래의 코드처럼 검색할 수 있다.  
`gugunAll.find({'sido_nm': sido_nm, 'gugun_nm': gugun_nm}, {'_id': 0}) # db.gugunAll.find({'sido_nm': '서울', 'gugun_nm': '강남구'}, {'_id': 0})`

사용하는 조건이 `eq`와 `and`만 사용하지 않기 때문에 아래의 코드도 사용할 수 있어야 한다.
```
starbucks> db.gugunAll.find(
  {
  	$and: [
    	{'sido_nm': {$eq: '서울'}},
    	{'gugun_nm': {$eq: '강남구'}}
  	]
  }, 
  {'_id': 0}
)
```

경기도의 경우 고양시, 성남시, 수원시, 안산시는 시를 구성하는 구가 존재하므로 고양시, 성남시, 수원시, 안산시만 입력하면 입력한 내용으로 `시작`하는 모든 구군 정보를 `정규식`을 사용해서 얻어올 수 있다.

```
starbucks> db.gugunAll.find(
  {
  	$and: [
    	{'sido_nm': {$eq: '경기'}},
    	{'gugun_nm': {$regex: '^성남시'}}
  	]
  }, 
  {'_id': 0}
)
```

In [6]:
gugun = gugunAll.find(
    {
        '$and': [
            {'sido_nm': {'$eq': sido_nm}},
            {'gugun_nm': {'$regex': '^' + gugun_nm}}
        ]
    }, 
    {'_id': 0}
)

# for gu in gugun:
    # print(gu)
    # print(gu['sido_cd'], gu['gugun_cd'])

시도 코드와 구군 코드를 이용해서 매장 정보를 크롤링해서 MongoDB에 저장하기 위해서 도큐먼트로 만든다.

In [7]:
targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'

storeList = []
for gu in gugun:
    response = requests.post(url=targetSite, data={
        'ins_lat': 37.365,
        'ins_lng': 127.0639,
        'p_sido_cd': gu['sido_cd'],
        'p_gugun_cd': gu['gugun_cd'],
        'in_biz_cd': '',
        'iend': 2300,
        'set_date': ''
    }).json()

    # print(response['list'][0])
    print(f"{gu['sido_nm']} {gu['gugun_nm']}의 스타벅스 매장은 {len(response['list'])}개 입니다.")
    time.sleep(1)
    
    for res in response['list']:
        dic = {
            'sido_code': res['sido_code'], # 시도코드
            'sido_name': res['sido_name'], # 시도이름
            'gugun_code': res['gugun_code'], # 구군코드
            'gugun_name': res['gugun_name'], # 구군이름
            's_name': res['s_name'], # 지점명
            'doro_address': res['doro_address'], # 도로명주소
            'lat': res['lat'], # 위도
            'lot': res['lot'], # 경도
        }
        storeList.append(dic)
    # ===== for res
# =====for gu

경기 성남시 분당구의 스타벅스 매장은 47개 입니다.
경기 성남시 수정구의 스타벅스 매장은 14개 입니다.
경기 성남시 중원구의 스타벅스 매장은 3개 입니다.


In [8]:
print(len(storeList))

64


도큐먼트를 MongoDB에 저장한다.

In [9]:
storeInfo = starbucks['storeInfo'] # starbucks> db.createCollection('storeInfo')
storeInfo.delete_many({}) # starbucks> db.storeInfo.deleteMany({})
for store in storeList:
    storeInfo.insert_one(store)

MongoDB에 저장된 구군별 스타벅스 매장 정보를 얻어온다.

In [10]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
starbucks = conn['starbucks']
storeInfo = starbucks['storeInfo']
storeList = storeInfo.find({}, {'_id': 0})

for store in storeList:
    print(store)

{'sido_code': '08', 'sido_name': '경기', 'gugun_code': '0811', 'gugun_name': '성남시 분당구', 's_name': '야탑역사거리', 'doro_address': '경기도 성남시 분당구 야탑로 102 (야탑동)', 'lat': '37.40969287889087', 'lot': '127.13030621693153'}
{'sido_code': '08', 'sido_name': '경기', 'gugun_code': '0811', 'gugun_name': '성남시 분당구', 's_name': '분당궁내DT', 'doro_address': '경기도 성남시 분당구 대왕판교로 302 (궁내동)', 'lat': '37.370636', 'lot': '127.102071'}
{'sido_code': '08', 'sido_name': '경기', 'gugun_code': '0811', 'gugun_name': '성남시 분당구', 's_name': '분당이매촌', 'doro_address': '경기도 성남시 분당구 양현로94번길 9 (이매동)', 'lat': '37.39161230854134', 'lot': '127.12552917726928'}
{'sido_code': '08', 'sido_name': '경기', 'gugun_code': '0811', 'gugun_name': '성남시 분당구', 's_name': '수내양지마을', 'doro_address': '경기도 성남시 분당구 내정로 167 (수내동)', 'lat': '37.373902337772996', 'lot': '127.11934378838681'}
{'sido_code': '08', 'sido_name': '경기', 'gugun_code': '0811', 'gugun_name': '성남시 분당구', 's_name': '분당아름마을', 'doro_address': '경기도 성남시 분당구 판교로 440 (이매동)', 'lat': '37.4051192990572', 'l

# 시도별 스타벅스 지점을 MongoDB에 저장하기

In [11]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
starbucks = conn['starbucks']
gugunAll = starbucks['gugunAll']
gugunList = gugunAll.find({}, {'_id': 0})

sidoSet = set()
for gugun in gugunList:
    sidoSet.add(gugun['sido_nm'])

print('시도 이름')
for sido in sidoSet:
    print(f'[{sido}]', end=' ')
print()
sido_nm = input('매장 정보를 검색하려는 시도 이름을 입력하세요: ')

gugun = gugunAll.find(
    {'sido_nm': {'$regex': sido_nm}},
    {'_id': 0}
).next()

targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
storeList = []
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': gugun['sido_cd'],
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
}).json()
print(f"{gugun['sido_nm']}의 스타벅스 매장은 {len(response['list'])}개 입니다.")

for res in response['list']:
    dic = {
        'sido_code': res['sido_code'],
        'sido_name': res['sido_name'],
        'gugun_code': res['gugun_code'],
        'gugun_name': res['gugun_name'],
        's_name': res['s_name'],
        'doro_address': res['doro_address'],
        'lat': res['lat'],
        'lot': res['lot'],
    }
    storeList.append(dic)

sidoAll = starbucks['sidoAll']
sidoAll.delete_many({})
for store in storeList:
    sidoAll.insert_one(store)

시도 이름
[경기] [제주] [대전] [울산] [부산] [전남광주] [서울] [충북] [세종] [인천] [경북대구통합특별시] [충남] [강원] [경남] [전북] 
매장 정보를 검색하려는 시도 이름을 입력하세요: 서울
서울의 스타벅스 매장은 714개 입니다.


# 전국 스타벅스 지점을 MongoDB에 저장하기

In [12]:
conn = pymongo.MongoClient('mongodb://localhost:27017')
starbucks = conn['starbucks']

targetSite = 'https://www.starbucks.co.kr/store/getStore.do?r=P4K5ZAPLE6'
storeList = []
response = requests.post(url=targetSite, data={
    'ins_lat': 37.365,
    'ins_lng': 127.0639,
    'p_sido_cd': '',
    'p_gugun_cd': '',
    'in_biz_cd': '',
    'iend': 2300,
    'set_date': ''
}).json()
print(f"전국의 스타벅스 매장은 {len(response['list'])}개 입니다.")

for res in response['list']:
    dic = {
        'sido_code': res['sido_code'], 'sido_name': res['sido_name'], 'gugun_code': res['gugun_code'], 'gugun_name': res['gugun_name'],
        's_name': res['s_name'], 'doro_address': res['doro_address'], 'lat': res['lat'], 'lot': res['lot'],
    }
    storeList.append(dic)

storeAll = starbucks['storeAll']
storeAll.delete_many({})
for store in storeList:
    storeAll.insert_one(store)

전국의 스타벅스 매장은 2213개 입니다.
